In [2]:
from pathlib import Path
from google.cloud import bigquery

PROJECT_ID = "customerretentionintelligence"
LOCATION = "US"
MAX_BYTES = 1_073_741_824  # Límite de 1 GiB para esta consulta.

root = Path.cwd()
if not (root / "sql").is_dir():
    root = root.parent

sql_path = root / "sql" / "08_build_customer_snapshots_v1.sql"
assert sql_path.is_file(), f"No encontré el archivo: {sql_path}"

sql = sql_path.read_text(encoding="utf-8").strip().rstrip(";")
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

check_sql = f"""
SELECT
    snapshot_date,
    split,
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS customer_count,
    IF(split = 'test', NULL, SUM(reorder_90d)) AS positive_cases,
    COUNTIF(max_feature_order_date > snapshot_date) AS future_order_features,
    COUNTIF(max_feature_item_date > snapshot_date) AS future_item_features,
    COUNTIF(reorder_90d IS NULL OR reorder_90d NOT IN (0, 1)) AS invalid_labels,
    COUNTIF(label_end_date != DATE_ADD(snapshot_date, INTERVAL 90 DAY)) AS invalid_horizons
FROM ({sql}) AS tad
GROUP BY snapshot_date, split
ORDER BY snapshot_date
"""

job_config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES)
summary = client.query(
    check_sql,
    job_config=job_config,
    location=LOCATION,
).to_dataframe()

display(summary)

,snapshot_date,split,row_count,customer_count,positive_cases,future_order_features,future_item_features,invalid_labels,invalid_horizons
0,2025-08-31,train,51526,51526,3064,0,0,0,0
1,2025-11-30,train,56438,56438,3551,0,0,0,0
2,2026-03-01,validation,61955,61955,4247,0,0,0,0
3,2026-06-01,test,68484,68484,<NA>,0,0,0,0


In [3]:
checks = [
    "future_order_features",
    "future_item_features",
    "invalid_labels",
    "invalid_horizons",
]

problems = summary.loc[
    (summary["row_count"] != summary["customer_count"])
    | (summary[checks] != 0).any(axis=1)
]

assert problems.empty, (
    "Hay controles que revisar:\n"
    + problems.to_string(index=False)
)

print("Controles aprobados en los cuatro snapshots.")

Controles aprobados en los cuatro snapshots.


In [ ]:
from google.api_core.exceptions import NotFound

TABLE_ID = f"{PROJECT_ID}.retention_ml.customer_snapshots_v1"
BUILD_TABLE = False  # Cambiar a True solo para la primera creación.

try:
    existing_table = client.get_table(TABLE_ID)
except NotFound:
    existing_table = None

if existing_table is not None:
    print(f"La tabla ya existe: {TABLE_ID}")
    print(f"Filas actuales: {existing_table.num_rows}")
    print("No se ejecutó ninguna escritura.")
elif not BUILD_TABLE:
    print(f"Destino disponible: {TABLE_ID}")
    print("No se creó la tabla. Revisa el nombre antes de cambiar BUILD_TABLE a True.")
else:
    build_config = bigquery.QueryJobConfig(
        destination=TABLE_ID,
        write_disposition=bigquery.WriteDisposition.WRITE_EMPTY,
        time_partitioning=bigquery.TimePartitioning(
            type_=bigquery.TimePartitioningType.DAY,
            field="snapshot_date",
        ),
        clustering_fields=["user_id"],
        maximum_bytes_billed=MAX_BYTES,
    )

    build_job = client.query(
        sql,
        job_config=build_config,
        location=LOCATION,
    )
    build_job.result()

    created_table = client.get_table(TABLE_ID)
    print(f"Tabla creada: {TABLE_ID}")
    print(f"Filas: {created_table.num_rows}")
    print(f"Job ID: {build_job.job_id}")

Tabla creada: customerretentionintelligence.retention_ml.customer_snapshots_v1
Filas: 238403
Job ID: 24268cac-f5fc-40b0-ac07-42086ff9c421


In [6]:
audit_sql = f"""
SELECT
    snapshot_date,
    split,
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS customer_count,
    IF(split = 'test', NULL, SUM(reorder_90d)) AS positive_cases,
    COUNTIF(max_feature_order_date > snapshot_date) AS future_order_features,
    COUNTIF(max_feature_item_date > snapshot_date) AS future_item_features,
    COUNTIF(
        reorder_90d = 1
        AND (
            first_future_order_date IS NULL
            OR first_future_order_date <= snapshot_date
            OR first_future_order_date > label_end_date
        )
    ) AS invalid_positive_dates,
    COUNTIF(
        reorder_90d = 0
        AND first_future_order_date IS NOT NULL
    ) AS invalid_negative_dates
FROM `{TABLE_ID}`
GROUP BY snapshot_date, split
ORDER BY snapshot_date
"""

audit = client.query(
    audit_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES),
    location=LOCATION,
).to_dataframe()

display(audit)

,snapshot_date,split,row_count,customer_count,positive_cases,future_order_features,future_item_features,invalid_positive_dates,invalid_negative_dates
0,2025-08-31,train,51526,51526,3064,0,0,0,0
1,2025-11-30,train,56438,56438,3551,0,0,0,0
2,2026-03-01,validation,61955,61955,4247,0,0,0,0
3,2026-06-01,test,68484,68484,<NA>,0,0,0,0


In [7]:
checks = [
    "future_order_features",
    "future_item_features",
    "invalid_positive_dates",
    "invalid_negative_dates",
]

assert (audit["row_count"] == audit["customer_count"]).all()
assert (audit[checks] == 0).all().all(), audit[checks].to_string(index=False)

print("Tabla guardada: cuatro snapshots sin duplicados ni errores temporales.")

Tabla guardada: cuatro snapshots sin duplicados ni errores temporales.


In [8]:
import pandas as pd

FEATURES = [
    "recency_days",
    "orders_history",
    "orders_90d",
    "orders_365d",
    "customer_age_days",
    "ordered_value_365d",
    "avg_order_value_365d",
]
TARGET = "reorder_90d"

training_sql = f"""
SELECT
    user_id,
    snapshot_date,
    split,
    {", ".join(FEATURES)},
    {TARGET}
FROM `{TABLE_ID}`
WHERE split IN ('train', 'validation')
  AND snapshot_date IN (
      DATE '2025-08-31',
      DATE '2025-11-30',
      DATE '2026-03-01'
  )
"""

df = client.query(
    training_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES),
    location=LOCATION,
).to_dataframe()

train = df.loc[df["split"] == "train"].copy()
valid = df.loc[df["split"] == "validation"].copy()

assert len(train) == 51526 + 56438
assert len(valid) == 61955
assert set(df["split"]) == {"train", "validation"}

X_train = train[FEATURES].astype("float64")
y_train = train[TARGET].astype("int64")
X_valid = valid[FEATURES].astype("float64")
y_valid = valid[TARGET].astype("int64")

assert set(y_train.unique()) == {0, 1}
assert set(y_valid.unique()) == {0, 1}

print("Entrenamiento:", X_train.shape, "| Positivos:", y_train.sum())
print("Validación:", X_valid.shape, "| Positivos:", y_valid.sum())
print("Nulos por variable:")
display(X_train.isna().sum().to_frame("train_nulos"))

Entrenamiento: (107964, 7) | Positivos: 6615
Validación: (61955, 7) | Positivos: 4247
Nulos por variable:


,train_nulos
recency_days,0
orders_history,0
orders_90d,0
orders_365d,0
customer_age_days,0
ordered_value_365d,0
avg_order_value_365d,60710


In [9]:
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        solver="lbfgs",
        C=1.0,
        max_iter=2000,
    )),
])

model.fit(X_train, y_train)
p_valid = model.predict_proba(X_valid)[:, 1]

dummy = DummyClassifier(strategy="prior")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
p_dummy = dummy.predict_proba(
    np.zeros((len(y_valid), 1))
)[:, 1]

assert np.isfinite(p_valid).all()
assert ((p_valid >= 0) & (p_valid <= 1)).all()

print("Primer modelo entrenado.")
print("Clientes evaluados en validación:", len(p_valid))
print("Tasa positiva de entrenamiento:", round(y_train.mean(), 4))

Primer modelo entrenado.
Clientes evaluados en validación: 61955
Tasa positiva de entrenamiento: 0.0613


In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

results = []

for name, scores in [
    ("dummy", p_dummy),
    ("logistic", p_valid),
]:
    results.append({
        "model": name,
        "average_precision": average_precision_score(y_valid, scores),
        "roc_auc": roc_auc_score(y_valid, scores),
        "brier": brier_score_loss(y_valid, scores),
    })

# Menos días desde la última orden = mayor puntuación de recompra.
recency_score = -X_valid["recency_days"].to_numpy()

results.append({
    "model": "recency_rule",
    "average_precision": average_precision_score(y_valid, recency_score),
    "roc_auc": roc_auc_score(y_valid, recency_score),
    "brier": np.nan,  # Es una puntuación, no una probabilidad.
})

metrics = pd.DataFrame(results)

print("Tasa positiva en validación:", round(y_valid.mean(), 4))
display(metrics.round(4))

Tasa positiva en validación: 0.0685


,model,average_precision,roc_auc,brier
0,dummy,0.0685,0.5000,0.0639
1,logistic,0.1186,0.6492,0.0628
2,recency_rule,0.1114,0.6408,NaN


In [11]:
import numpy as np
import pandas as pd

y = y_valid.to_numpy()
top_n = int(np.ceil(len(y) * 0.10))
base_rate = y.mean()

results_top10 = []

for name, scores in [
    ("logistic", p_valid),
    ("recency_rule", -X_valid["recency_days"].to_numpy()),
]:
    top_idx = np.argsort(-np.asarray(scores), kind="stable")[:top_n]
    top_rate = y[top_idx].mean()

    results_top10.append({
        "model": name,
        "selected_customers": top_n,
        "repurchase_rate_top10": top_rate,
        "lift_vs_average": top_rate / base_rate,
    })

print(f"Tasa general de recompra: {base_rate:.2%}")
display(pd.DataFrame(results_top10).round(3))

Tasa general de recompra: 6.85%


,model,selected_customers,repurchase_rate_top10,lift_vs_average
0,logistic,6196,0.151,2.208
1,recency_rule,6196,0.140,2.037


In [12]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_estimators=250,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.9,
    reg_lambda=5,
    random_state=42,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)
p_xgb = xgb_model.predict_proba(X_valid)[:, 1]

y = y_valid.to_numpy()
top_n = int(np.ceil(len(y) * 0.10))
base_rate = y.mean()

comparison = []

for name, scores in [
    ("logistic", p_valid),
    ("xgboost", p_xgb),
]:
    top_idx = np.argsort(-scores, kind="stable")[:top_n]

    comparison.append({
        "model": name,
        "average_precision": average_precision_score(y, scores),
        "roc_auc": roc_auc_score(y, scores),
        "brier": brier_score_loss(y, scores),
        "repurchase_top10": y[top_idx].mean(),
        "lift_top10": y[top_idx].mean() / base_rate,
    })

display(pd.DataFrame(comparison).round(4))

,model,average_precision,roc_auc,brier,repurchase_top10,lift_top10
0,logistic,0.1186,0.6492,0.0628,0.1514,2.2084
1,xgboost,0.1389,0.6791,0.0621,0.1635,2.3850


In [13]:
import numpy as np
import pandas as pd

y = y_valid.to_numpy()
order = np.argsort(p_xgb, kind="stable")
groups = np.array_split(order, 10)

calibration_rows = []

for group_number, idx in enumerate(groups, start=1):
    predicted = p_xgb[idx].mean() * 100
    observed = y[idx].mean() * 100

    calibration_rows.append({
        "group": group_number,
        "customers": len(idx),
        "predicted_pct": predicted,
        "observed_pct": observed,
        "gap_pp": observed - predicted,
    })

calibration = pd.DataFrame(calibration_rows)

print(
    f"Promedio predicho: {p_xgb.mean():.2%} | "
    f"Tasa real: {y.mean():.2%}"
)
display(calibration.round(2))

Promedio predicho: 6.12% | Tasa real: 6.85%


,group,customers,predicted_pct,observed_pct,gap_pp
0,1,6196,1.95,1.76,-0.19
1,2,6196,3.28,3.37,0.09
2,3,6196,3.87,4.16,0.30
3,4,6196,4.56,4.41,-0.16
4,5,6196,4.94,4.71,-0.23
5,6,6195,5.69,6.15,0.46
6,7,6195,6.71,7.04,0.33
7,8,6195,7.85,8.26,0.42
8,9,6195,9.36,12.33,2.97
9,10,6195,13.04,16.35,3.32


In [14]:
rows = []

for name, scores in [("logistic", p_valid), ("xgboost", p_xgb)]:
    top_idx = np.argsort(-scores, kind="stable")[:top_n]

    rows.append({
        "model": name,
        "predicted_overall_pct": scores.mean() * 100,
        "observed_overall_pct": y.mean() * 100,
        "predicted_top10_pct": scores[top_idx].mean() * 100,
        "observed_top10_pct": y[top_idx].mean() * 100,
    })

display(pd.DataFrame(rows).round(2))

,model,predicted_overall_pct,observed_overall_pct,predicted_top10_pct,observed_top10_pct
0,logistic,6.15,6.85,10.78,15.14
1,xgboost,6.12,6.85,13.04,16.35


In [15]:
gain = xgb_model.get_booster().get_score(importance_type="gain")

importance = pd.DataFrame({
    "feature": X_train.columns,
    "gain": [gain.get(feature, 0) for feature in X_train.columns],
}).sort_values("gain", ascending=False)

display(importance.round(2))

,feature,gain
3,orders_365d,53.28
1,orders_history,32.00
0,recency_days,16.72
5,ordered_value_365d,7.30
4,customer_age_days,6.13
6,avg_order_value_365d,3.76
2,orders_90d,3.28


In [16]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    xgb_model,
    X_valid,
    y_valid,
    scoring="average_precision",
    n_repeats=3,
    random_state=42,
    n_jobs=1,
)

perm_table = pd.DataFrame({
    "feature": X_valid.columns,
    "ap_drop": perm.importances_mean,
    "variation": perm.importances_std,
}).sort_values("ap_drop", ascending=False)

display(perm_table.round(4))

,feature,ap_drop,variation
1,orders_history,0.0363,0.0010
4,customer_age_days,0.0225,0.0023
0,recency_days,0.0173,0.0020
3,orders_365d,0.0049,0.0003
2,orders_90d,0.0012,0.0001
5,ordered_value_365d,0.0002,0.0005
6,avg_order_value_365d,-0.0028,0.0005


In [17]:
from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

features_6 = [
    col for col in X_train.columns
    if col != "avg_order_value_365d"
]

xgb_6 = clone(xgb_model)
xgb_6.fit(X_train[features_6], y_train)
p_xgb_6 = xgb_6.predict_proba(X_valid[features_6])[:, 1]

ablation_rows = []

for name, scores in [
    ("xgboost_7_features", p_xgb),
    ("xgboost_6_features", p_xgb_6),
]:
    top_idx = np.argsort(-scores, kind="stable")[:top_n]

    ablation_rows.append({
        "model": name,
        "average_precision": average_precision_score(y_valid, scores),
        "roc_auc": roc_auc_score(y_valid, scores),
        "brier": brier_score_loss(y_valid, scores),
        "repurchase_top10": y_valid.to_numpy()[top_idx].mean(),
    })

display(pd.DataFrame(ablation_rows).round(4))

,model,average_precision,roc_auc,brier,repurchase_top10
0,xgboost_7_features,0.1389,0.6791,0.0621,0.1635
1,xgboost_6_features,0.1415,0.6795,0.0621,0.1683


### Selección provisional del modelo

El objetivo de `order_created_v1` es predecir si un cliente creará una nueva orden durante los 90 días posteriores a cada fecha de corte. Entrené con los snapshots del 2025-08-31 y 2025-11-30 y comparé los modelos en el snapshot de validación del 2026-03-01, sin utilizar el test.

Elegí provisionalmente XGBoost con seis variables: `recency_days`, `orders_history`, `orders_90d`, `orders_365d`, `customer_age_days` y `ordered_value_365d`. En validación obtuvo una *average precision* de 0,1415, ROC AUC de 0,6795 y una tasa de nuevas órdenes de 16,83 % en el 10 % mejor puntuado, frente al 6,85 % general. La mejora respecto a la versión de siete variables fue pequeña, por lo que no la considero concluyente.

El modelo permite ordenar clientes según su propensión a crear una nueva orden; esto no demuestra el efecto de una campaña de retención. Todavía debo revisar la calibración de esta versión de seis variables. El snapshot de test del 2026-06-01 continúa reservado para una evaluación final.


In [18]:
order_6 = np.argsort(p_xgb_6, kind="stable")
groups_6 = np.array_split(order_6, 10)
y = y_valid.to_numpy()

calibration_6 = pd.DataFrame([
    {
        "group": number,
        "customers": len(idx),
        "predicted_pct": p_xgb_6[idx].mean() * 100,
        "observed_pct": y[idx].mean() * 100,
        "gap_pp": (y[idx].mean() - p_xgb_6[idx].mean()) * 100,
    }
    for number, idx in enumerate(groups_6, start=1)
])

print(
    f"Promedio predicho: {p_xgb_6.mean():.2%} | "
    f"Tasa real: {y.mean():.2%}"
)
display(calibration_6.round(2))

Promedio predicho: 6.13% | Tasa real: 6.85%


,group,customers,predicted_pct,observed_pct,gap_pp
0,1,6196,1.96,1.76,-0.20
1,2,6196,3.27,3.20,-0.08
2,3,6196,3.89,4.34,0.45
3,4,6196,4.56,4.34,-0.22
4,5,6196,4.95,4.70,-0.25
5,6,6195,5.70,5.94,0.24
6,7,6195,6.70,7.44,0.74
7,8,6195,7.86,8.25,0.39
8,9,6195,9.38,11.75,2.37
9,10,6195,13.01,16.84,3.83


In [19]:
validation_label_sql = """
WITH validation_rows AS (
    SELECT user_id, snapshot_date, reorder_90d
    FROM `customerretentionintelligence.retention_ml.customer_snapshots_v1`
    WHERE split = 'validation'
      AND snapshot_date = DATE '2026-03-01'
),
recomputed AS (
    SELECT
        v.user_id,
        IF(COUNT(o.order_id) > 0, 1, 0) AS recomputed_label
    FROM validation_rows AS v
    LEFT JOIN `customerretentionintelligence.retention_ml.clean_orders` AS o
        ON o.user_id = v.user_id
       AND DATE(o.created_at) > v.snapshot_date
       AND DATE(o.created_at) <= DATE_ADD(v.snapshot_date, INTERVAL 90 DAY)
    GROUP BY v.user_id
)
SELECT
    COUNT(*) AS rows_checked,
    COUNTIF(v.reorder_90d != r.recomputed_label) AS mismatches,
    SUM(v.reorder_90d) AS stored_positives,
    SUM(r.recomputed_label) AS recomputed_positives
FROM validation_rows AS v
JOIN recomputed AS r USING (user_id)
"""

label_check = client.query(
    validation_label_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES),
    location=LOCATION,
).to_dataframe()

display(label_check)

assert label_check.loc[0, "rows_checked"] == 61955
assert label_check.loc[0, "mismatches"] == 0

,rows_checked,mismatches,stored_positives,recomputed_positives
0,61955,0,4247,4247


In [20]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

test_sql = """
WITH test_rows AS (
    SELECT
        user_id,
        snapshot_date,
        recency_days,
        orders_history,
        orders_90d,
        orders_365d,
        customer_age_days,
        ordered_value_365d,
        avg_order_value_365d
    FROM `customerretentionintelligence.retention_ml.customer_snapshots_v1`
    WHERE split = 'test'
      AND snapshot_date = DATE '2026-06-01'
),
test_labels AS (
    SELECT
        t.user_id,
        IF(COUNT(o.order_id) > 0, 1, 0) AS reorder_90d
    FROM test_rows AS t
    LEFT JOIN `customerretentionintelligence.retention_ml.clean_orders` AS o
        ON o.user_id = t.user_id
       AND DATE(o.created_at) > t.snapshot_date
       AND DATE(o.created_at) <= DATE_ADD(t.snapshot_date, INTERVAL 90 DAY)
    GROUP BY t.user_id
)
SELECT
    t.*,
    l.reorder_90d
FROM test_rows AS t
JOIN test_labels AS l USING (user_id)
"""

test_data = client.query(
    test_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES),
    location=LOCATION,
).to_dataframe()

assert len(test_data) == 68484
assert test_data["user_id"].is_unique
assert test_data["snapshot_date"].astype(str).eq("2026-06-01").all()
assert test_data["reorder_90d"].isin([0, 1]).all()

y_test = test_data["reorder_90d"].astype("int64").to_numpy()

# Ambos modelos quedan exactamente como se entrenaron.
X_test_6 = test_data[features_6].astype("float64")
X_test_logistic = test_data[X_train.columns].astype("float64")

p_test_xgb = xgb_6.predict_proba(X_test_6)[:, 1]
p_test_logistic = model.predict_proba(X_test_logistic)[:, 1]
p_test_dummy = np.full(len(y_test), y_train.mean())
recency_test = -test_data["recency_days"].to_numpy()

top_n_test = int(np.ceil(len(y_test) * 0.10))
base_rate_test = y_test.mean()

test_results = []

for name, scores in [
    ("dummy", p_test_dummy),
    ("recency_rule", recency_test),
    ("logistic_7f", p_test_logistic),
    ("xgboost_6f", p_test_xgb),
]:
    top_idx = np.argsort(-scores, kind="stable")[:top_n_test]
    top_rate = y_test[top_idx].mean()

    test_results.append({
        "model": name,
        "average_precision": average_precision_score(y_test, scores),
        "roc_auc": roc_auc_score(y_test, scores),
        "brier": (
            np.nan if name == "recency_rule"
            else brier_score_loss(y_test, scores)
        ),
        "repurchase_top10": top_rate,
        "lift_top10": top_rate / base_rate_test,
    })

print(
    f"Test: {len(y_test):,} clientes | "
    f"{y_test.sum():,} positivos | "
    f"tasa real: {base_rate_test:.2%}"
)
display(pd.DataFrame(test_results).round(4))

Test: 68,484 clientes | 5,604 positivos | tasa real: 8.18%


,model,average_precision,roc_auc,brier,repurchase_top10,lift_top10
0,dummy,0.0818,0.5000,0.0756,0.1581,1.9324
1,recency_rule,0.1409,0.6612,NaN,0.1707,2.0858
2,logistic_7f,0.1566,0.6727,0.0737,0.1946,2.3784
3,xgboost_6f,0.1969,0.7113,0.0724,0.2170,2.6514


In [21]:
test_summary = pd.DataFrame(test_results).copy()
test_summary.loc[
    test_summary["model"] == "dummy",
    ["repurchase_top10", "lift_top10"]
] = np.nan

display(test_summary.round(4))

,model,average_precision,roc_auc,brier,repurchase_top10,lift_top10
0,dummy,0.0818,0.5000,0.0756,NaN,NaN
1,recency_rule,0.1409,0.6612,NaN,0.1707,2.0858
2,logistic_7f,0.1566,0.6727,0.0737,0.1946,2.3784
3,xgboost_6f,0.1969,0.7113,0.0724,0.2170,2.6514


### Evaluación temporal del primer modelo

Para `order_created_v1` busco predecir si un cliente creará una nueva orden durante los 90 días posteriores a una fecha de corte. Esto mide creación de órdenes, no ventas completadas. Entrené con los snapshots del 2025-08-31 y 2025-11-30, utilicé el 2026-03-01 para validación y reservé el 2026-06-01 para una única evaluación final.

En validación seleccioné XGBoost con seis variables: `recency_days`, `orders_history`, `orders_90d`, `orders_365d`, `customer_age_days` y `ordered_value_365d`. Obtuvo una *average precision* de 0,1415 y un ROC AUC de 0,6795. En el 10 % mejor puntuado, el 16,83 % creó otra orden, frente al 6,85 % general. La mejora al quitar la séptima variable fue pequeña, por lo que no concluyo que esa variable sea inútil en cualquier otro período.

Antes de evaluar el test, comprobé que la lógica de la etiqueta reproducía exactamente los 61.955 registros de validación: hubo 0 diferencias. En el test, de 68.484 clientes elegibles, 5.604 crearon otra orden (8,18 %). El XGBoost seleccionado obtuvo una *average precision* de 0,1969, ROC AUC de 0,7113 y Brier de 0,0724. Superó a la regresión logística y a la regla basada solo en recencia. Entre el 10 % mejor puntuado, la tasa de nueva orden fue 21,70 %, equivalente a 2,65 veces la tasa general del test.

Estos resultados muestran capacidad para **ordenar clientes según su propensión a crear una nueva orden**, no el efecto que tendría contactarlos con una campaña. En validación, el modelo subestimó la recompra en los grupos de mayor puntuación; por ello no considero aún calibradas sus probabilidades. El test ya fue utilizado para la evaluación final y no lo usaré para volver a elegir variables o ajustar este modelo.
